<a href="https://colab.research.google.com/github/Echinel/Echinel/blob/claude%2Fhel-churn-mitigation-framework-fZpOK/NHS_AMDARI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import numpy as np
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

# Load datasets
hospitals = pd.read_csv('Hospital.csv')
medicines = pd.read_csv('Medicine.csv')
activity = pd.read_csv('Hospital_Activity.csv')
demand = pd.read_csv('Medicine_Demand.csv')

# **Merge Hospital metadata and Activity into the main Demand table**

In [18]:
# Reload demand to ensure a clean state for merging each time this cell is run
demand = pd.read_csv('Medicine_Demand.csv')

# Step 1: Merge 'hospitals' into 'demand' on 'Hospital_ID'
# This adds static hospital info (Name, Type, Region, Beds) and some static aggregate activity (Admissions, Bed_Occupancy).
# The activity data from 'activity.csv' is time-series specific and will override these.
demand = pd.merge(demand, hospitals, on='Hospital_ID', how='left')

# Step 2: Ensure 'Date' columns are in datetime format for accurate merging
demand['Date'] = pd.to_datetime(demand['Date'])
activity['Date'] = pd.to_datetime(activity['Date'])

# Step 3: Merge 'activity' into the combined 'demand' DataFrame on 'Hospital_ID' and 'Date'
# By default, for overlapping columns, it will suffix the left (demand, from hospitals) with _x and right (activity) with _y.
demand = pd.merge(demand, activity, on=['Hospital_ID', 'Date'], how='left')

# Step 4: Merge 'medicines' into 'demand' on 'Medicine_ID'
# This adds Medicine_Name, Category, Form, Shelf_Life_Months
demand = pd.merge(demand, medicines, on='Medicine_ID', how='left')

# Step 5: Rename columns to desired final names and resolve conflicts
demand = demand.rename(columns={
    'Hospital_Name': 'hospital_name',
    'Medicine_Name': 'medicine_name',
    'Shelf_Life_Months': 'shelf_life', # Rename for feature_cols
    'Admissions_y': 'Admissions', # Prefer Admissions from activity_y
    'Emergency_Admissions_y': 'Emergency_Admissions', # Prefer Emergency_Admissions from activity_y
    'Bed_Occupancy_y': 'Bed_Occupancy', # Prefer Bed_Occupancy from activity_y
    'Hospital_Type_y': 'Hospital_Type' # Assuming Hospital_Type ended up as Hospital_Type_y based on previous output
})

# Drop duplicated/unused columns if they exist after renaming
demand.drop(columns=[
    'Admissions_x',
    'Emergency_Admissions_x',
    'Bed_Occupancy_x',
    'Region_x',
    'Beds_x',
    'Region_y',
    'Beds_y' # Drop other suffixed columns that are not needed or are redundant
], inplace=True, errors='ignore') # Use errors='ignore' for robustness

# Display the first few rows of the merged DataFrame to verify
display(demand.head())

,Date,Hospital_ID,Medicine_ID,Quantity_Dispensed,Current_Stock,Unit_Cost,Emergency_Orders,Medicine_Waste,hospital_name,Region,Hospital_Type,Beds,Admissions,Emergency_Admissions,Bed_Occupancy,medicine_name,Category,Form,shelf_life
0,2021-01-01,H01,M001,2036,409,0.1495,0,0,Northbridge General Hospital,West Yorkshire,General,850,1360,544,78.1,Amoxicillin 500mg,Antibiotic,Capsule,24
1,2021-02-01,H01,M001,2252,986,0.1433,0,0,Northbridge General Hospital,West Yorkshire,General,850,1275,531,75.4,Amoxicillin 500mg,Antibiotic,Capsule,24
2,2021-03-01,H01,M001,2385,1840,0.1435,0,0,Northbridge General Hospital,West Yorkshire,General,850,1246,371,71.4,Amoxicillin 500mg,Antibiotic,Capsule,24
3,2021-04-01,H01,M001,1743,97,0.1554,0,0,Northbridge General Hospital,West Yorkshire,General,850,1197,490,72.5,Amoxicillin 500mg,Antibiotic,Capsule,24
4,2021-05-01,H01,M001,1494,2011,0.1482,0,0,Northbridge General Hospital,West Yorkshire,General,850,1078,369,68.1,Amoxicillin 500mg,Antibiotic,Capsule,24


# **Create Key Performance Features**

In [19]:
# Step 1: Create 'Value_Dispensed' column
demand['Value_Dispensed'] = demand['Quantity_Dispensed'] * demand['Unit_Cost']

# Step 2: Create 'Safety Stock indicator'
# This assumes an emergency order is implied when quantity dispensed exceeds current stock.
demand['Safety_Stock_Indicator'] = (demand['Quantity_Dispensed'] > demand['Current_Stock']).astype(int)

# Display the first few rows of the updated DataFrame with new features
display(demand.head())

,Date,Hospital_ID,Medicine_ID,Quantity_Dispensed,Current_Stock,Unit_Cost,Emergency_Orders,Medicine_Waste,hospital_name,Region,...,Beds,Admissions,Emergency_Admissions,Bed_Occupancy,medicine_name,Category,Form,shelf_life,Value_Dispensed,Safety_Stock_Indicator
0,2021-01-01,H01,M001,2036,409,0.1495,0,0,Northbridge General Hospital,West Yorkshire,...,850,1360,544,78.1,Amoxicillin 500mg,Antibiotic,Capsule,24,304.3820,1
1,2021-02-01,H01,M001,2252,986,0.1433,0,0,Northbridge General Hospital,West Yorkshire,...,850,1275,531,75.4,Amoxicillin 500mg,Antibiotic,Capsule,24,322.7116,1
2,2021-03-01,H01,M001,2385,1840,0.1435,0,0,Northbridge General Hospital,West Yorkshire,...,850,1246,371,71.4,Amoxicillin 500mg,Antibiotic,Capsule,24,342.2475,1
3,2021-04-01,H01,M001,1743,97,0.1554,0,0,Northbridge General Hospital,West Yorkshire,...,850,1197,490,72.5,Amoxicillin 500mg,Antibiotic,Capsule,24,270.8622,1
4,2021-05-01,H01,M001,1494,2011,0.1482,0,0,Northbridge General Hospital,West Yorkshire,...,850,1078,369,68.1,Amoxicillin 500mg,Antibiotic,Capsule,24,221.4108,0


# **Phase 2A: The SARIMAX Baseline Pipeline**

In [20]:
def train_eval_sarimax(df, hospital_name, medicine_name):
    """
    Filters data for a specific hospital-medicine pair, fits a SARIMAX model,
    and returns evaluation metrics + forecasts.
    """
    # 1. Isolate the target time series
    # Ensure 'Date' column is used for filtering and indexing
    subset = df[(df['hospital_name'] == hospital_name) & (df['medicine_name'] == medicine_name)].copy()
    subset = subset.sort_values('Date').set_index('Date') # Corrected 'date' to 'Date'

    # Fill any missing dates if they exist (safety check, though panel is complete)
    subset = subset.asfreq('MS')

    # 2. Split into Train (48 months) and Test (12 months)
    # Total dataset spans Jan 2021 to Dec 2025 (60 months)
    train = subset.iloc[:-12]
    test = subset.iloc[-12:]

    # Define Target (y) and Exogenous features (X)
    y_train = train['Quantity_Dispensed']
    X_train = train[['Admissions', 'Bed_Occupancy']] # Using hospital activity as exogenous factors

    y_test = test['Quantity_Dispensed']
    X_test = test[['Admissions', 'Bed_Occupancy']]

    # 3. Fit SARIMAX Baseline
    # Order (p,d,q) and Seasonal Order (P,D,Q,s)
    # Using standard defaults: (1,1,1) x (1,1,0,12) assuming annual seasonality
    try:
        model = SARIMAX(y_train,
                        exog=X_train,
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 0, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)

        results = model.fit(disp=False)

        # 4. Forecast the Test Period
        predictions = results.forecast(steps=12, exog=X_test)
        predictions.index = y_test.index

        # 5. Evaluate Performance
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        # Avoid zero division with epsilon protection
        mape = np.mean(np.abs((y_test - predictions) / np.clip(y_test, 1e-5, None))) * 100

        return {
            'model_summary': results,
            'predictions': predictions,
            'actuals': y_test,
            'metrics': {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
        }

    except Exception as e:
        print(f"Failed to fit model for {hospital_name} - {medicine_name}: {e}")
        return None

# Example execution for a single pair to test the engine
# example_results = train_eval_sarimax(df, 'Northbridge General', 'Amoxicillin')
# print(example_results['metrics'])

# **Phase 2B: Transitioning to XGBoost**

In [21]:
def build_xgboost_features(df):
    """
    Transforms the panel data into a supervised ML matrix
    by creating lag and rolling window features.
    """
    # Explicitly select base columns needed for features or as features themselves
    base_cols = [
        'Hospital_ID', 'Medicine_ID', 'Date', 'Quantity_Dispensed',
        'Medicine_Waste', 'Hospital_Type', 'Category', 'Form',
        'Admissions', 'Bed_Occupancy', 'Unit_Cost', 'shelf_life'
    ]

    # Ensure these columns exist in the input df
    # This also acts as a safeguard against previous merge errors
    for col in base_cols:
        if col not in df.columns:
            raise KeyError(f"Missing required base column in input DataFrame for build_xgboost_features: {col}")

    features_df = df[base_cols].copy().sort_values(['Hospital_ID', 'Medicine_ID', 'Date'])

    # 1. Time Lags (What was dispensed 1, 2, and 3 months ago)
    for lag in [1, 2, 3]:
        features_df[f'demand_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].shift(lag)
        features_df[f'waste_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Medicine_Waste'].shift(lag)

    # 2. Rolling Statistics (Capturing recent trajectory)
    features_df['demand_rolling_mean_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).mean())
    features_df['demand_rolling_std_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).std())

    # 3. Categorical Encoding
    # Convert text categories into numerical codes for XGBoost
    features_df['hospital_type_code'] = features_df['Hospital_Type'].astype('category').cat.codes
    features_df['med_category_code'] = features_df['Category'].astype('category').cat.codes
    features_df['med_form_code'] = features_df['Form'].astype('category').cat.codes

    # Drop rows with NaN caused by lags (these will be the first few months for each group)
    features_df = features_df.dropna()

    return features_df

# Prepare the data matrix
# ml_matrix = build_xgboost_features(df)

# **The Model Comparison Pipeline**

In [22]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

def build_xgboost_features(df):
    """
    Transforms the panel data into a supervised ML matrix
    by creating lag and rolling window features.
    """
    # Explicitly select base columns needed for features or as features themselves
    base_cols = [
        'Hospital_ID', 'Medicine_ID', 'Date', 'Quantity_Dispensed',
        'Medicine_Waste', 'Hospital_Type', 'Category', 'Form',
        'Admissions', 'Bed_Occupancy', 'Unit_Cost', 'shelf_life',
        'hospital_name', 'medicine_name' # Added these columns
    ]

    features_df = df[base_cols].copy().sort_values(['Hospital_ID', 'Medicine_ID', 'Date'])

    # 1. Time Lags (What was dispensed 1, 2, and 3 months ago)
    for lag in [1, 2, 3]:
        features_df[f'demand_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].shift(lag)
        features_df[f'waste_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Medicine_Waste'].shift(lag)

    # 2. Rolling Statistics (Capturing recent trajectory)
    features_df['demand_rolling_mean_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).mean())
    features_df['demand_rolling_std_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).std())

    # 3. Categorical Encoding
    # Convert text categories into numerical codes for XGBoost
    features_df['hospital_type_code'] = features_df['Hospital_Type'].astype('category').cat.codes
    features_df['med_category_code'] = features_df['Category'].astype('category').cat.codes
    features_df['med_form_code'] = features_df['Form'].astype('category').cat.codes

    # Drop rows with NaN caused by lags (these will be the first few months for each group)
    features_df = features_df.dropna()

    return features_df

# 1. Prepare Data Matrix for Machine Learning
ml_df = build_xgboost_features(demand)

# Define feature columns and target
feature_cols = [
    'Admissions', 'Bed_Occupancy', 'Unit_Cost', 'shelf_life',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3',
    'waste_lag_1', 'waste_lag_2', 'waste_lag_3',
    'demand_rolling_mean_3m', 'demand_rolling_std_3m',
    'hospital_type_code', 'med_category_code', 'med_form_code'
]
target_col = 'Quantity_Dispensed'

# 2. Time-Series Split (Last 12 months for Testing: Jan 2025 - Dec 2025)
# Using structural splitting to prevent look-ahead bias
train_mask = ml_df['Date'] < '2025-01-01'
test_mask = ml_df['Date'] >= '2025-01-01'

X_train, y_train = ml_df.loc[train_mask, feature_cols], ml_df.loc[train_mask, target_col]
X_test, y_test = ml_df.loc[test_mask, feature_cols], ml_df.loc[test_mask, target_col]

# Keep historical tracking metadata for evaluations
test_meta = ml_df.loc[test_mask, ['hospital_name', 'medicine_name', 'Date', target_col]].copy()

# 3. Train the Global XGBoost Model
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

# Predict using XGBoost
test_meta['xgb_pred'] = xgb_model.predict(X_test)

# 4. Generate SARIMAX Predictions for the Matchup
# (Iterative processing over unique pairs for the identical test window)
sarimax_preds = []

unique_pairs = demand[['hospital_name', 'medicine_name']].drop_duplicates()

print("Calculating SARIMAX baselines across the panel (this may take a moment)...")
for _, row in unique_pairs.iterrows():
    h_name = row['hospital_name']
    m_name = row['medicine_name']

    # Run our previously built SARIMAX function
    sarimax_res = train_eval_sarimax(demand, h_name, m_name)

    if sarimax_res is not None:
        # Ensure predictions index matches y_test index from SARIMAX function
        pred_series = sarimax_res['predictions'].reset_index()
        pred_series.columns = ['Date', 'sarimax_pred'] # Ensure consistent column name
        pred_series['hospital_name'] = h_name
        pred_series['medicine_name'] = m_name
        sarimax_preds.append(pred_series)

# Merge SARIMAX forecasts back into our master evaluation matrix
if sarimax_preds:
    sarimax_df = pd.concat(sarimax_preds, ignore_index=True)
    evaluation_master = test_meta.merge(sarimax_df, on=['hospital_name', 'medicine_name', 'Date'], how='inner')

    # 5. Compute Comparative Performance Metrics
    def calculate_metrics(actual, predicted):
        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        # Avoid zero division with epsilon protection
        mape = np.mean(np.abs((actual - predicted) / np.clip(actual, 1e-5, None))) * 100
        return mae, rmse, mape

    xgb_mae, xgb_rmse, xgb_mape = calculate_metrics(evaluation_master['Quantity_Dispensed'], evaluation_master['xgb_pred'])
    sar_mae, sar_rmse, sar_mape = calculate_metrics(evaluation_master['Quantity_Dispensed'], evaluation_master['sarimax_pred'])

    # 6. Output Performance Leaderboard
    comparison_table = pd.DataFrame({
        'Metric': ['MAE (Units Required)', 'RMSE (Variance Penalty)', 'MAPE (Percentage Error)'],
        'SARIMAX Baseline': [sar_mae, sar_rmse, f"{sar_mape:.2f}%"],
        'XGBoost Regressor': [xgb_mae, xgb_rmse, f"{xgb_mape:.2f}%"],

    }).set_index('Metric')

    print("\n=== Trust-wide Model Evaluation Results ===")
    print(comparison_table)
else:
    print("No SARIMAX predictions were generated.")

Calculating SARIMAX baselines across the panel (this may take a moment)...

=== Trust-wide Model Evaluation Results ===
                        SARIMAX Baseline XGBoost Regressor
Metric                                                    
MAE (Units Required)           59.986196         69.841988
RMSE (Variance Penalty)       117.446432        136.967085
MAPE (Percentage Error)           11.33%            13.78%


# **Phase 3: Inventory Logic Pipeline**

In [23]:
# 1. Prepare base DataFrame for replenishment by merging necessary info
replenishment_df = evaluation_master.copy()

# Merge in Current_Stock, Unit_Cost, and shelf_life for the test period
# from the original 'demand' DataFrame.
# Need to select only the relevant columns from 'demand' for merging
demand_for_merge = demand[['Date', 'hospital_name', 'medicine_name', 'Current_Stock', 'Unit_Cost', 'shelf_life']].copy()

replenishment_df = pd.merge(
    replenishment_df,
    demand_for_merge,
    on=['Date', 'hospital_name', 'medicine_name'],
    how='left'
)

# Rename original Quantity_Dispensed from evaluation_master to actual_dispensed for clarity
replenishment_df = replenishment_df.rename(columns={'Quantity_Dispensed': 'actual_dispensed'})

# 2. Define safety stock and target stock logic
# Simple safety stock: 20% of predicted monthly demand
replenishment_df['safety_stock'] = replenishment_df['sarimax_pred'] * 0.2
# Target stock: predicted demand + safety stock
replenishment_df['target_stock'] = replenishment_df['sarimax_pred'] + replenishment_df['safety_stock']

# 3. Calculate Projected End-of-Month Stock
# This is Current_Stock at the beginning of the month minus predicted demand
replenishment_df['projected_stock_eom'] = replenishment_df['Current_Stock'] - replenishment_df['sarimax_pred']

# 4. Calculate Replenishment Quantity
# If projected stock falls below 0 (or a reorder point, here we use 0), order enough to reach target stock.
# The 'reorder_point' could be more sophisticated, but starting with 0 for simplicity.
replenishment_df['replenishment_needed'] = np.maximum(0, replenishment_df['target_stock'] - replenishment_df['projected_stock_eom'])

# 5. Calculate Estimated Replenishment Cost
replenishment_df['replenishment_cost'] = replenishment_df['replenishment_needed'] * replenishment_df['Unit_Cost']

# 6. Create Replenishment Ledger - display relevant columns
replenishment_ledger = replenishment_df[[
    'Date', 'hospital_name', 'medicine_name', 'actual_dispensed',
    'sarimax_pred', 'Current_Stock', 'projected_stock_eom',
    'replenishment_needed', 'replenishment_cost', 'shelf_life'
]].copy()

# Display the first few rows of the replenishment ledger
print("Replenishment Ledger Sample:")
display(replenishment_ledger.head())

# Optional: Summarize total replenishment cost for the period
total_cost = replenishment_ledger['replenishment_cost'].sum()
print(f"\nTotal Estimated Replenishment Cost for the Test Period: ${total_cost:,.2f}")

# Optional: Add summary stats for replenishment needs
print("\nReplenishment Needs Summary:")
display(replenishment_ledger[replenishment_ledger['replenishment_needed'] > 0].describe())

Replenishment Ledger Sample:


,Date,hospital_name,medicine_name,actual_dispensed,sarimax_pred,Current_Stock,projected_stock_eom,replenishment_needed,replenishment_cost,shelf_life
0,2025-01-01,Northbridge General Hospital,Amoxicillin 500mg,3075,2391.147806,187,-2204.147806,5073.525173,886.344848,24
1,2025-02-01,Northbridge General Hospital,Amoxicillin 500mg,2257,2220.414120,2999,778.585880,1885.911064,308.723641,24
2,2025-03-01,Northbridge General Hospital,Amoxicillin 500mg,1822,2228.323937,1177,-1051.323937,3725.312662,635.910871,24
3,2025-04-01,Northbridge General Hospital,Amoxicillin 500mg,1437,1531.878257,1843,311.121743,1527.132165,240.065176,24
4,2025-05-01,Northbridge General Hospital,Amoxicillin 500mg,1231,1596.330611,612,-984.330611,2899.927343,480.517961,24



Total Estimated Replenishment Cost for the Test Period: $7,914,487.98

Replenishment Needs Summary:


,Date,actual_dispensed,sarimax_pred,Current_Stock,projected_stock_eom,replenishment_needed,replenishment_cost,shelf_life
count,4272,4272.000000,4272.000000,4272.000000,4272.000000,4272.000000,4272.000000,4272.000000
mean,2025-06-16 15:11:27.640449536,567.621489,568.179191,292.336142,-275.843049,957.658079,1852.642317,30.337079
min,2025-01-01 00:00:00,12.000000,7.791511,0.000000,-4498.323465,6.785808,1.094711,12.000000
25%,2025-03-01 00:00:00,140.750000,142.086898,31.000000,-334.245385,227.089381,29.293967,24.000000
50%,2025-07-01 00:00:00,298.500000,304.988931,111.000000,-116.346997,493.884721,110.993735,36.000000
75%,2025-10-01 00:00:00,698.500000,712.298421,326.000000,-32.901469,1177.096651,727.086514,36.000000
max,2025-12-01 00:00:00,7202.000000,7353.187379,5203.000000,1508.650887,11429.634198,87873.101058,60.000000
std,NaN,695.995660,687.613267,487.437455,480.096924,1212.515605,6383.801383,8.359442


# **Phase 4: The Interactive Dash Application Interface**

In [24]:
!pip install streamlit pandas plotly scikit-learn xgboost statsmodels
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

Next, I'll generate the Streamlit application. This code will contain all the logic for data loading, preprocessing, model inference, and the dashboard layout.

In [25]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# --- 1. Data Loading and Preprocessing ---
@st.cache_data
def load_data():
    hospitals = pd.read_csv('Hospital.csv')
    medicines = pd.read_csv('Medicine.csv')
    activity = pd.read_csv('Hospital_Activity.csv')
    demand = pd.read_csv('Medicine_Demand.csv')

    demand = pd.merge(demand, hospitals, on='Hospital_ID', how='left')
    demand['Date'] = pd.to_datetime(demand['Date'])
    activity['Date'] = pd.to_datetime(activity['Date'])
    demand = pd.merge(demand, activity, on=['Hospital_ID', 'Date'], how='left')
    demand = pd.merge(demand, medicines, on='Medicine_ID', how='left')

    demand = demand.rename(columns={
        'Hospital_Name': 'hospital_name',
        'Medicine_Name': 'medicine_name',
        'Shelf_Life_Months': 'shelf_life',
        'Admissions_y': 'Admissions',
        'Emergency_Admissions_y': 'Emergency_Admissions',
        'Bed_Occupancy_y': 'Bed_Occupancy',
        'Hospital_Type_y': 'Hospital_Type'
    })

    demand.drop(columns=[
        'Admissions_x',
        'Emergency_Admissions_x',
        'Bed_Occupancy_x',
        'Region_x',
        'Beds_x',
        'Region_y',
        'Beds_y'
    ], inplace=True, errors='ignore')

    demand['Value_Dispensed'] = demand['Quantity_Dispensed'] * demand['Unit_Cost']
    demand['Safety_Stock_Indicator'] = (demand['Quantity_Dispensed'] > demand['Current_Stock']).astype(int)

    return demand

@st.cache_data
def build_xgboost_features(df):
    base_cols = [
        'Hospital_ID', 'Medicine_ID', 'Date', 'Quantity_Dispensed',
        'Medicine_Waste', 'Hospital_Type', 'Category', 'Form',
        'Admissions', 'Bed_Occupancy', 'Unit_Cost', 'shelf_life',
        'hospital_name', 'medicine_name' # Add these for merging later
    ]

    features_df = df[base_cols].copy().sort_values(['Hospital_ID', 'Medicine_ID', 'Date'])

    for lag in [1, 2, 3]:
        features_df[f'demand_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].shift(lag)
        features_df[f'waste_lag_{lag}'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Medicine_Waste'].shift(lag)

    features_df['demand_rolling_mean_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).mean())
    features_df['demand_rolling_std_3m'] = features_df.groupby(['Hospital_ID', 'Medicine_ID'])['Quantity_Dispensed'].transform(lambda x: x.shift(1).rolling(3).std())

    features_df['hospital_type_code'] = features_df['Hospital_Type'].astype('category').cat.codes
    features_df['med_category_code'] = features_df['Category'].astype('category').cat.codes
    features_df['med_form_code'] = features_df['Form'].astype('category').cat.codes

    return features_df.dropna()

@st.cache_data
def train_eval_sarimax(df, hospital_name, medicine_name):
    subset = df[(df['hospital_name'] == hospital_name) & (df['medicine_name'] == medicine_name)].copy()
    subset = subset.sort_values('Date').set_index('Date')
    subset = subset.asfreq('MS')

    train = subset.iloc[:-12]
    test = subset.iloc[-12:]

    y_train = train['Quantity_Dispensed']
    X_train = train[['Admissions', 'Bed_Occupancy']]
    y_test = test['Quantity_Dispensed']
    X_test = test[['Admissions', 'Bed_Occupancy']]

    try:
        model = SARIMAX(y_train,
                        exog=X_train,
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 0, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)

        results = model.fit(disp=False)
        predictions = results.forecast(steps=12, exog=X_test)
        predictions.index = y_test.index

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mape = np.mean(np.abs((y_test - predictions) / np.clip(y_test, 1e-5, None))) * 100

        return {
            'predictions': predictions,
            'actuals': y_test,
            'metrics': {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
        }

    except Exception as e:
        st.warning(f"Failed to fit SARIMAX model for {hospital_name} - {medicine_name}: {e}")
        return None

# --- 2. Streamlit App Layout ---
st.set_page_config(layout="wide", page_title="NHS Pharmacy Demand Intelligence")

st.markdown("<h1 style='text-align: center; color: #005EB8;'>Northbridge Healthcare NHS Trust</h1>", unsafe_allow_html=True)
st.markdown("<h3 style='text-align: center; color: #333;'>Pharmacy Inventory Optimization & Predictive Analytics Platform</h3>", unsafe_allow_html=True)

demand_df = load_data()
ml_df = build_xgboost_features(demand_df)

hospitals_list = demand_df['hospital_name'].unique().tolist()
medicines_list = demand_df['medicine_name'].unique().tolist()

# --- Sidebar for Filters ---
st.sidebar.header("Dashboard Filters")
selected_hospital = st.sidebar.selectbox("Select Hospital Facility:", hospitals_list)
selected_medicine = st.sidebar.selectbox("Select Medical Item:", medicines_list)

# --- 3. Interactive Callback Pipeline Engine ---

# Filter data for the selected hospital and medicine
pair_df_filtered = ml_df[(ml_df["hospital_name"] == selected_hospital) & (ml_df["medicine_name"] == selected_medicine)].copy()
original_demand_pair = demand_df[(demand_df["hospital_name"] == selected_hospital) & (demand_df["medicine_name"] == selected_medicine)].copy()

if not pair_df_filtered.empty:
    pair_df_filtered = pair_df_filtered.sort_values("Date")

    # Split data for training/testing (last 12 months for test)
    train_slice = pair_df_filtered.iloc[:-12]
    test_slice = pair_df_filtered.iloc[-12:]

    # Check if there's enough data for SARIMAX training
    sarimax_res = train_eval_sarimax(demand_df, selected_hospital, selected_medicine)

    sarimax_preds = None
    sar_mae = np.nan
    sar_rmse = np.nan

    if sarimax_res:
        sarimax_preds = sarimax_res['predictions']
        sar_mae = sarimax_res['metrics']['MAE']
        sar_rmse = sarimax_res['metrics']['RMSE']

    # XGBoost Model
    feature_cols = [
        'Admissions', 'Bed_Occupancy', 'Unit_Cost', 'shelf_life',
        'demand_lag_1', 'demand_lag_2', 'demand_lag_3',
        'waste_lag_1', 'waste_lag_2', 'waste_lag_3',
        'demand_rolling_mean_3m', 'demand_rolling_std_3m',
        'hospital_type_code', 'med_category_code', 'med_form_code'
    ]

    # Filter features for columns that actually exist in the training data
    available_features = [col for col in feature_cols if col in train_slice.columns]

    xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    xgb_model.fit(train_slice[available_features], train_slice["Quantity_Dispensed"])
    xgb_preds = xgb_model.predict(test_slice[available_features])

    actuals = test_slice["Quantity_Dispensed"].values
    xgb_mae = mean_absolute_error(actuals, xgb_preds)
    xgb_rmse = np.sqrt(mean_squared_error(actuals, xgb_preds))

    # --- Replenishment Logic ---
    last_row = original_demand_pair.sort_values('Date').iloc[-1]
    rmse_production = sar_rmse # Using SARIMAX RMSE for operational calculations
    next_month_forecast = xgb_preds[-1]

    z_score = 2.58 if last_row["Category"] in ["Antibiotics", "Insulin"] else 1.96
    lead_time = 0.5 # Assuming average lead time of 0.5 months

    safety_stock = int(z_score * rmse_production * np.sqrt(lead_time))
    reorder_point = int((next_month_forecast * lead_time) + safety_stock)
    current_stock_val = int(last_row["Current_Stock"])

    # Economic Order Quantity (EOQ) calculation - simplified
    # Holding cost assumed as 20% of unit cost per year
    # Ordering cost assumed as $25 per order
    ordering_cost = 25.0
    holding_cost_rate = 0.20
    annual_demand_estimate = next_month_forecast * 12 # Estimate annual demand

    if last_row["Unit_Cost"] > 0:
        eoq = int(np.sqrt((2 * ordering_cost * annual_demand_estimate) / (last_row["Unit_Cost"] * holding_cost_rate)))
    else:
        eoq = 0 # Cannot calculate EOQ if unit cost is zero

    # Max expiry limit: ensuring we don't order more than can be used before expiry
    # Assuming medicines should be used 3 months before shelf_life ends
    max_expiry_limit = int(next_month_forecast * (last_row["shelf_life"] - 3))
    optimized_order = min(eoq, max_expiry_limit) if max_expiry_limit > 0 else eoq

    if current_stock_val < reorder_point:
        action_directive = f"CRITICAL REQUIREMENT: Trigger order sequence for {optimized_order} units immediately."
    else:
        action_directive = "STABLE BOUNDARY: Warehouse stock levels running healthy. Hold acquisitions."

    # --- Dashboard Elements ---
    st.subheader(f"Demand Trends for {selected_medicine} at {selected_hospital}")

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=pair_df_filtered["Date"], y=pair_df_filtered["Quantity_Dispensed"], name="Actual Demand Run", line=dict(color="#212529", width=2)))
    if sarimax_preds is not None:
        fig.add_trace(go.Scatter(x=test_slice["Date"], y=sarimax_preds, name="SARIMAX Baseline Forecast", line=dict(color="#005EB8", dash="dash")))
    fig.add_trace(go.Scatter(x=test_slice["Date"], y=xgb_preds, name="XGBoost Champion Model", line=dict(color="#D9381E", width=2)))

    fig.update_layout(
        template="plotly_white",
        margin=dict(l=40, r=40, t=20, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis_title="Timeline Multi-Year Index", yaxis_title="Quantity Dispensed (Units)"
    )
    st.plotly_chart(fig, use_container_width=True)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Model Performance Metric Matrix")
        metrics_data = [
            {"Model Framework": "SARIMAX Statistical Baseline", "MAE (Units)": f"{sar_mae:.2f}", "RMSE (Variance Penalty)": f"{sar_rmse:.2f}"},
            {"Model Framework": "XGBoost Regressor (Champion)", "MAE (Units)": f"{xgb_mae:.2f}", "RMSE (Variance Penalty)": f"{xgb_rmse:.2f}"}
        ]
        st.dataframe(pd.DataFrame(metrics_data).set_index("Model Framework"))

    with col2:
        st.subheader("Calculated Next-Month Operational Stock Directives")
        replenishment_data = [
            {"Metric Indicator": "Current Warehouse Asset Balance", "Calculated Volume": current_stock_val, "Operational Strategy / Notes": "Live stock check count"},
            {"Metric Indicator": "Dynamic System Reorder Point (ROP)", "Calculated Volume": reorder_point, "Operational Strategy / Notes": f"Trigger line floor includes {safety_stock} units Safety Stock safety net"},
            {"Metric Indicator": "Recommended Order Placement", "Calculated Volume": optimized_order if current_stock_val < reorder_point else 0, "Operational Strategy / Notes": action_directive}
        ]
        st.dataframe(pd.DataFrame(replenishment_data).set_index("Metric Indicator"), height=250)
else:
    st.warning("No data available for the selected hospital and medicine combination.")

Overwriting app.py


Now, run the Streamlit app. It will provide a public URL you can click to access your dashboard.

In [28]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://pretty-rings-kiss.loca.lt
2026-07-22 18:38:38.219 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.247.248:8501

  Stopping...
^C
